# 4. Feature Engineering, Normalization & Baseline Models

**Input:** the clean tables from notebook 2 (`data/clean/reports_clean.parquet`, `case_ingredients/`).
**Output:** the modeling inputs in `data/features/` and baseline scores in `data/features/baseline_metrics.json`.

**Task:** predict `y_hosp_or_death` for one case from the patient, the report and the suspect drugs.
The feature decisions come from the EDA findings table at the end of notebook 3 ("finding N" below refers to it).

| Part | What it answers | Sections |
|---|---|---|
| **A. Modeling population & split** | Which rows go into the model, and how to split without leakage? | 2 |
| **B. Feature engineering** | Which columns does the classifier see? | 3 |
| **C. Normalization & encoding** | On what scale? | 4 |
| **D. Baseline models** | What score does any later model have to beat? | 5 |

## 1. Setup and load

Only the case table and the suspect-ingredient table are needed. Reactions are not used: they leak the target (finding 9).

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

reports = pd.read_parquet(CLEAN_DIR / "reports_clean.parquet")
print(f"reports: {reports.shape[0]:,} rows x {reports.shape[1]} cols, "
      f"{reports.memory_usage(deep=True).sum() / 1e9:.2f} GB")

TARGET = "y_hosp_or_death"
# float copy of the target: 1.0 / 0.0 / NaN
reports["y"] = reports[TARGET].astype("float32")

# folder of part files; ingredient as a categorical keeps the 14M rows small
ingredients = pq.read_table(CLEAN_DIR / "case_ingredients", columns=["safetyreportid", "ingredient"],
                            read_dictionary=["ingredient"]).to_pandas()
print(f"ingredients: {len(ingredients):,} rows, {ingredients['ingredient'].nunique():,} distinct")

reports: 8,477,316 rows x 23 cols, 0.84 GB
ingredients: 14,011,496 rows, 7,183 distinct


# Part A — Modeling population and split

## 2. Which rows, which split

**Population** (decisions from the EDA findings):
- known target only
- dropped **legacy cases** (first received before 2020): they resurface because serious cases get follow-ups (finding 14)
- dropped **mass reports**: the group size is computed across all years, so the flag would leak future reports (finding 15)

**Splitting by FAERS quarter**, so the model is always evaluated on reports from after its training period:

| Split | Quarters | Use |
|---|---|---|
| train | 2020Q1 – 2024Q3 | fit everything: model, scaler, top-K drug list, category levels |
| validation | 2025Q1 – 2025Q2 | tune hyperparameters, pick thresholds |
| test | 2025Q3 – 2026Q2 | touch once, at the end |

2023Q3, 2024Q1 and 2024Q4 are absent from the data (notebook 1), so 2024Q4 is not a missing gap in the split itself.

**Training sample:** a 5 GB machine cannot fit most models on 5M+ rows. We take a 1,000,000-row sample of train,
stratified on the target. Validation and test are kept whole.

In [2]:
TRAIN_END, VAL_END = "2024Q3", "2025Q2"
TRAIN_SAMPLE_SIZE = 1_000_000
RANDOM_STATE = 42

pop_steps = [("all cases", len(reports))]
mask = reports["y"].notna()
pop_steps.append(("known target", int(mask.sum())))
mask &= ~reports["is_legacy_case"]
pop_steps.append(("not legacy", int(mask.sum())))
mask &= ~reports["is_mass_report"]
pop_steps.append(("not mass report", int(mask.sum())))
display(pd.DataFrame(pop_steps, columns=["step", "cases"]).set_index("step").T)

pop = reports.loc[mask].reset_index(drop=True)
pop["split"] = pd.Categorical(
    np.select([pop["quarter"] <= TRAIN_END, pop["quarter"] <= VAL_END], ["train", "val"], "test"),
    categories=["train", "val", "test"])

train_rows = pop.index[pop["split"] == "train"]
frac = TRAIN_SAMPLE_SIZE / len(train_rows)
sample_rows = (pop.loc[train_rows].groupby("y", group_keys=False)
               .sample(frac=frac, random_state=RANDOM_STATE).index)
pop["in_train_sample"] = False
pop.loc[sample_rows, "in_train_sample"] = True

split_summary = pd.DataFrame({
    name: {"cases": int(rows.sum()), "positive_rate": round(float(pop.loc[rows, "y"].mean()), 3),
           "quarters": f"{pop.loc[rows, 'quarter'].min()} - {pop.loc[rows, 'quarter'].max()}"}
    for name, rows in [("train", pop["split"] == "train"), ("train sample", pop["in_train_sample"]),
                       ("val", pop["split"] == "val"), ("test", pop["split"] == "test")]
}).T
display(split_summary)

pop[["safetyreportid", "quarter", "split", "in_train_sample"]].to_parquet(FEATURES_DIR / "split.parquet", index=False)
print(f"saved {FEATURES_DIR / 'split.parquet'}")

step,all cases,known target,not legacy,not mass report
cases,8477316,8380159,8126921,8012423


,cases,positive_rate,quarters
train,5914363,0.258,2020Q1 - 2024Q3
train sample,1000000,0.258,2020Q1 - 2024Q3
val,626361,0.26,2025Q1 - 2025Q2
test,1471699,0.262,2025Q3 - 2026Q2


saved /Users/suvidhinikhil/Data science projects/Milestone 2 projects/FAERS_Milestone2_Final/data/features/split.parquet


# Part B — Feature engineering

## 3. Features

One row per case. Everything a category level or a top-K list depends on is learned from the **train split**.

| Group | Features | Finding (nb 3) |
|---|---|---|
| Patient | `age_years`, `age_missing`, `age_group`, `sex` (incl. `missing`), `weight_kg`, `weight_missing` | 3, 4, 5 |
| Report | `reporter`, `report_type`, `is_us`, `country` (top 10 occurrence countries in train, else `OTHER`) | 11, 12, 13 |
| Drugs | `n_drugs`, `n_suspect_ingredients`, `n_other_ingredients` (outside top-K), top-K ingredient multi-hot | 6, 7 |

**Excluded on purpose:** any reaction term (finding 9), `expedited` (10), `safetyreportversion`, receipt date / lag,
`is_legacy_case` (14), dedup flags (15), and `death` / `hospitalization` / `is_serious` (they are the target).

In [3]:
TOP_K_DRUGS = 400        # ~80% of ingredient mentions (notebook 3, section 4)
TOP_N_COUNTRIES = 10

train = pop["split"] == "train"
country = pop["occur_country"].astype("string")
top_countries = country[train].value_counts().head(TOP_N_COUNTRIES).index.tolist()


def missing_label(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("missing")


base = pd.DataFrame({
    "age_years": pop["age_years"].astype("float32"),
    "age_missing": pop["age_years"].isna().astype("int8"),
    "age_group": missing_label(pop["age_group"]),
    "sex": missing_label(pop["sex"]),
    "weight_kg": pop["weight_kg"].astype("float32"),
    "weight_missing": pop["weight_kg"].isna().astype("int8"),
    "reporter": missing_label(pop["reporter"]),
    "report_type": missing_label(pop["report_type"]),
    "is_us": country.eq("US").fillna(False).astype("int8"),
    "country": country.where(country.isin(top_countries), "OTHER").fillna("missing"),
    "n_drugs": pop["n_drugs"].astype("int32"),
    "n_suspect_ingredients": pop["n_suspect_ingredients"].astype("int32"),
})

# --- top-K ingredient multi-hot (sparse) ---
case_pos = pd.Index(pop["safetyreportid"])
ing_pop = ingredients.loc[ingredients["safetyreportid"].isin(case_pos), ["safetyreportid", "ingredient"]]
ing_pop = ing_pop.assign(row=case_pos.get_indexer(ing_pop["safetyreportid"]))
train_ing = ing_pop[train.to_numpy()[ing_pop["row"].to_numpy()]]
top_drugs = pd.Index(train_ing["ingredient"].value_counts().head(TOP_K_DRUGS).index.astype(str))
col = top_drugs.get_indexer(ing_pop["ingredient"].astype(str))
keep = col >= 0
drug_matrix = sp.csr_matrix((np.ones(int(keep.sum()), dtype=np.uint8), (ing_pop["row"].to_numpy()[keep], col[keep])),
                            shape=(len(pop), TOP_K_DRUGS))
drug_matrix.data[:] = 1  # a case lists each standardized ingredient once, but be safe
base["n_other_ingredients"] = (base["n_suspect_ingredients"] - np.asarray(drug_matrix.sum(axis=1)).ravel()).astype("int32")
drug_feature_names = [f"drug={d}" for d in top_drugs]

share_covered = (base["n_other_ingredients"] < base["n_suspect_ingredients"]).groupby(pop["split"], observed=True).mean()
print(f"top-{TOP_K_DRUGS} drugs learned on train; share of cases with >= 1 of them:")
display(share_covered.round(3).to_frame("share").T)
display(base.head())

# unscaled table: the right input for tree models (which need no scaling)
base_out = pd.concat([pop[["safetyreportid", "split", "in_train_sample", "y"]], base], axis=1)
base_out.to_parquet(FEATURES_DIR / "case_features.parquet", index=False)
sp.save_npz(FEATURES_DIR / "drug_multihot.npz", drug_matrix)
(FEATURES_DIR / "top_drugs.json").write_text(json.dumps(list(top_drugs)))
print("saved case_features.parquet (unscaled), drug_multihot.npz (row-aligned), top_drugs.json")

top-400 drugs learned on train; share of cases with >= 1 of them:


split,train,val,test
share,0.877,0.815,0.761


,age_years,age_missing,age_group,sex,weight_kg,weight_missing,reporter,report_type,is_us,country,n_drugs,n_suspect_ingredients,n_other_ingredients
0,NaN,1,missing,male,NaN,1,physician,spontaneous,1,US,1,1,1
1,48.000000,0,adult,missing,NaN,1,other_hcp,spontaneous,0,OTHER,16,1,1
2,85.000000,0,elderly,missing,NaN,1,other_hcp,spontaneous,0,OTHER,1,1,1
3,0.583333,0,infant,female,9.039,0,consumer,spontaneous,1,US,3,1,0
4,NaN,1,missing,missing,NaN,1,consumer,spontaneous,1,US,1,1,0


saved case_features.parquet (unscaled), drug_multihot.npz (row-aligned), top_drugs.json


# Part C — Normalization and encoding

## 4. Scaling and encoding

One scikit-learn `ColumnTransformer`, fit on the **train sample** only:

| Columns | Transform | Why |
|---|---|---|
| `age_years`, `weight_kg` | median impute → standard scale | missing indicators already capture "missing" |
| `n_drugs`, `n_suspect_ingredients`, `n_other_ingredients` | `log1p` → standard scale | heavy right tail (max 3,900) |
| `age_missing`, `weight_missing`, `is_us` | as is (0/1) | already on a 0–1 scale |
| `sex`, `age_group`, `reporter`, `report_type`, `country` | one-hot; levels under 1,000 train cases → "infrequent" | linear models need it |
| drug multi-hot | as is (0/1), appended as sparse columns | 400 binary columns |

Output per split: a sparse matrix `X_<split>.npz`, labels `y_<split>.npy`, case ids `ids_<split>.npy`. The fitted
transformer is saved, so later notebooks can apply it to new data and read the feature names.

In [4]:
NUM_COLS = ["age_years", "weight_kg"]
COUNT_COLS = ["n_drugs", "n_suspect_ingredients", "n_other_ingredients"]
FLAG_COLS = ["age_missing", "weight_missing", "is_us"]
CAT_COLS = ["sex", "age_group", "reporter", "report_type", "country"]

preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM_COLS),
        ("count", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                            ("scale", StandardScaler())]), COUNT_COLS),
        ("flag", "passthrough", FLAG_COLS),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=1000), CAT_COLS),
    ],
    sparse_threshold=1.0,  # always return a sparse matrix
)
fit_rows = pop["in_train_sample"].to_numpy()
preprocessor.fit(base.loc[fit_rows, NUM_COLS + COUNT_COLS + FLAG_COLS + CAT_COLS])
feature_names = list(preprocessor.get_feature_names_out()) + drug_feature_names

matrices = {}
for name, rows in [("train", fit_rows), ("val", (pop["split"] == "val").to_numpy()),
                   ("test", (pop["split"] == "test").to_numpy())]:
    X = sp.hstack([preprocessor.transform(base.loc[rows]), drug_matrix[rows]], format="csr", dtype=np.float32)
    y = pop.loc[rows, "y"].to_numpy(dtype=np.int8)
    sp.save_npz(FEATURES_DIR / f"X_{name}.npz", X)
    np.save(FEATURES_DIR / f"y_{name}.npy", y)
    np.save(FEATURES_DIR / f"ids_{name}.npy", pop.loc[rows, "safetyreportid"].to_numpy())
    matrices[name] = (X, y)
    print(f"{name:5s}: X {X.shape[0]:,} x {X.shape[1]}  ({X.nnz / X.shape[0]:.1f} non-zeros per row), positive rate {y.mean():.3f}")

joblib.dump({"preprocessor": preprocessor, "top_drugs": list(top_drugs), "top_countries": top_countries,
             "feature_names": feature_names}, FEATURES_DIR / "preprocessor.joblib")
(FEATURES_DIR / "feature_names.json").write_text(json.dumps(feature_names))

X_train_dense_part = matrices["train"][0][:, : len(feature_names) - TOP_K_DRUGS].toarray()
print("\nscaled train columns (first 8): mean / std")
display(pd.DataFrame({"mean": X_train_dense_part.mean(axis=0), "std": X_train_dense_part.std(axis=0)},
                     index=feature_names[: len(feature_names) - TOP_K_DRUGS]).head(8).round(3).T)
del X_train_dense_part

train: X 1,000,000 x 440  (13.3 non-zeros per row), positive rate 0.258


val  : X 626,361 x 440  (13.1 non-zeros per row), positive rate 0.260


test : X 1,471,699 x 440  (13.0 non-zeros per row), positive rate 0.262



scaled train columns (first 8): mean / std


,num__age_years,num__weight_kg,count__n_drugs,count__n_suspect_ingredients,count__n_other_ingredients,flag__age_missing,flag__weight_missing,flag__is_us
mean,-0.000,0.0,0.0,-0.000,-0.000,0.466,0.839,0.666
std,0.999,1.0,1.0,1.003,1.003,0.498,0.366,0.470


# Part D — Baseline models

## 5. Baselines

Three reference points, all fit on the train sample and scored on **validation** (test stays untouched):

| Model | Features | Why |
|---|---|---|
| Prior | none | ROC AUC 0.5 |
| Logistic regression | 40 patient / report / count columns only | how much do the non-drug features carry? |
| Logistic regression | all 440 (adds the top-400 drug flags) | the baseline to beat |


In [5]:
X_tr, y_tr = matrices["train"]
X_va, y_va = matrices["val"]
n_base = len(feature_names) - TOP_K_DRUGS  # columns before the drug flags

baselines = {
    "prior": (DummyClassifier(strategy="prior"), slice(None)),
    "logreg, no drugs": (LogisticRegression(max_iter=300), slice(0, n_base)),
    "logreg, all features": (LogisticRegression(max_iter=300), slice(None)),
}
results = {}
for name, (model, cols) in baselines.items():
    model.fit(X_tr[:, cols], y_tr)
    p_va = model.predict_proba(X_va[:, cols])[:, 1]
    results[name] = {"n_features": 0 if name == "prior" else X_tr[:, cols].shape[1],
                     "val_roc_auc": round(roc_auc_score(y_va, p_va), 4),
                     "val_avg_precision": round(average_precision_score(y_va, p_va), 4)}

results_df = pd.DataFrame(results).T
display(results_df)
print(f"validation positive rate: {y_va.mean():.3f}")

(FEATURES_DIR / "baseline_metrics.json").write_text(json.dumps(results, indent=2))
print("saved baseline_metrics.json")

,n_features,val_roc_auc,val_avg_precision
prior,0.0,0.5000,0.2599
"logreg, no drugs",40.0,0.7663,0.5081
"logreg, all features",440.0,0.7985,0.5507


validation positive rate: 0.260
saved baseline_metrics.json


**Which features drive the full logistic regression?** The largest coefficients should be readable: drugs with very low
or very high outcome rates in notebook 3, section 4.

In [6]:
coefs = pd.Series(baselines["logreg, all features"][0].coef_[0], index=feature_names).sort_values()
print("most negative / most positive coefficients:")
display(pd.concat([coefs.head(10), coefs.tail(10)]).round(2).to_frame("coef").T)

non_drug = coefs[~coefs.index.str.startswith("drug=")]
print("non-drug features, by absolute coefficient:")
display(non_drug.reindex(non_drug.abs().sort_values(ascending=False).index).head(15).round(2).to_frame("coef").T)

most negative / most positive coefficients:


,drug=ADAPALENE,drug=MINOXIDIL,drug=COPPER,drug=OLOPATADINE,drug=ISOTRETINOIN,drug=SODIUM CITRATE,drug=GALCANEZUMAB-GNLM,drug=EXENATIDE,drug=ETONOGESTREL,drug=PEGFILGRASTIM,drug=BOSENTAN,drug=AMPHETAMINE,drug=TEDUGLUTIDE,drug=VOXELOTOR,drug=CARBIDOPA,drug=DABIGATRAN ETEXILATE,drug=TAFAMIDIS,drug=PALIVIZUMAB,drug=ALCOHOL,drug=RIFAXIMIN
coef,-2.93,-2.47,-2.35,-2.13,-2.06,-1.93,-1.8,-1.75,-1.75,-1.65,1.23,1.27,1.29,1.36,1.38,1.39,1.53,1.76,1.96,2.12


non-drug features, by absolute coefficient:


,cat__report_type_infrequent_sklearn,cat__reporter_lawyer,cat__report_type_study,cat__country_FR,cat__country_DE,flag__is_us,cat__country_US,flag__age_missing,cat__reporter_consumer,cat__report_type_other,cat__reporter_missing,cat__country_IT,cat__country_AU,cat__reporter_pharmacist,cat__age_group_missing
coef,-0.9,-0.75,0.71,0.44,0.42,-0.42,-0.42,-0.4,-0.37,0.32,0.3,-0.28,-0.28,0.27,-0.25


## Handover

`data/features/`:

| File | For |
|---|---|
| `split.parquet` | which split / sample each case is in |
| `case_features.parquet` + `drug_multihot.npz` | unscaled features, for tree models (row-aligned) |
| `X_{train,val,test}.npz`, `y_*.npy`, `ids_*.npy` | scaled sparse matrices, for linear models / neural nets |
| `preprocessor.joblib`, `feature_names.json`, `top_drugs.json` | fitted transformer and feature metadata (fit on the train sample only) |
| `baseline_metrics.json` | validation scores of the three baselines: the numbers to beat |